<a href="https://colab.research.google.com/github/csiyu0962-collab/DataScience-GenAI-Submissions/blob/main/6_02_DNN_101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=1xqQczl0FG-qtNA2_WQYuWePW9oU8irqJ)

# 6.02 Dense Neural Network (with PyTorch)
This will expand on our logistic regression example and take us through building our first neural network. If you haven't already, be sure to check (and if neccessary) switch to GPU processing by clicking Runtime > Change runtime type and selecting GPU. We can test this has worked with the following code:

In [ ]:
import torch

# Check for GPU availability
print("Num GPUs Available: ", torch.cuda.device_count())

Num GPUs Available:  1


In [1]:
import torch

print(torch.cuda.is_available())

True


Hopefully your code shows you have 1 GPU available! Next let's get some data. We'll start with another in-built dataset:

In [2]:
# upload an in-built Python (OK semi-in-built) dataset
from sklearn.datasets import load_diabetes

import pandas as pd
import numpy as np

# import the data
data = load_diabetes()
data

{'data': array([[ 0.03807591,  0.05068012,  0.06169621, ..., -0.00259226,
          0.01990749, -0.01764613],
        [-0.00188202, -0.04464164, -0.05147406, ..., -0.03949338,
         -0.06833155, -0.09220405],
        [ 0.08529891,  0.05068012,  0.04445121, ..., -0.00259226,
          0.00286131, -0.02593034],
        ...,
        [ 0.04170844,  0.05068012, -0.01590626, ..., -0.01107952,
         -0.04688253,  0.01549073],
        [-0.04547248, -0.04464164,  0.03906215, ...,  0.02655962,
          0.04452873, -0.02593034],
        [-0.04547248, -0.04464164, -0.0730303 , ..., -0.03949338,
         -0.00422151,  0.00306441]], shape=(442, 10)),
 'target': array([151.,  75., 141., 206., 135.,  97., 138.,  63., 110., 310., 101.,
         69., 179., 185., 118., 171., 166., 144.,  97., 168.,  68.,  49.,
         68., 245., 184., 202., 137.,  85., 131., 283., 129.,  59., 341.,
         87.,  65., 102., 265., 276., 252.,  90., 100.,  55.,  61.,  92.,
        259.,  53., 190., 142.,  75., 142.

We are working on a regression problem, with "structured" data which has already been cleaned and normalised. We can skip the usual cleaning/engineering steps. However, we do need to get the data into PyTorch:

In [3]:
# Convert data to PyTorch tensors
X = torch.tensor(data.data, dtype=torch.float32)
y = torch.tensor(data.target, dtype=torch.float32).reshape(-1, 1) # Reshape y to be a column vector

Now our data is stored in tensors we can do train/test splitting as before (in fact we can use sklearn as before):

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch.Size([353, 10]) torch.Size([353, 1])
torch.Size([89, 10]) torch.Size([89, 1])


Now we can set up our batches for training. As we have a nice round 400 let's go with batches of 50 (8 batches in total). We'll also seperate the features and labels:

In [5]:
from torch.utils.data import TensorDataset, DataLoader

# Create TensorDatasets and DataLoaders
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=50, shuffle=False)

Now its time to build our model. We'll keep it simple ... a model with an input layer of 10 features and then 2x _Dense_ (fully connected) layers each with 5 neurons and ReLU activation. Our output layer will be size=1 given this is a regression problem and we want a single value output per prediction.

This will be easier to understand if you have read through the logistic regression tutorial.

In [6]:
import torch
import torch.nn as nn

# Define the model
class DiabetesModel(nn.Module):
    def __init__(self):
        super(DiabetesModel, self).__init__()
        # we'll set up the layers as a sequence using nn.Sequential
        self.layers = nn.Sequential(

            # first layer will be a linear layer that has 5x neurons
            # (5x sets of linear regression)
            # the layer takes the 10 features as input (i.e. 10, 5)
            nn.Linear(10, 5),

            nn.ReLU(), # ReLU activation

            # second linear layer again has 5 neurons
            # this time taking the input as the output of the last layer
            # (which had 5x neurons)
            nn.Linear(5, 5),

            nn.ReLU(), # ReLU again

            # last linear layer takes the output from the previous 5 neurons
            # this time its a single output with no activation
            # i.e. this is the predicitons (regression)
            nn.Linear(5, 1)
        )

    def forward(self, x):
        return self.layers(x) # pass the data through the layers

As before we need to create a model object, specify the loss (criterion) and an optimiser (which we cover next week):

In [7]:
import torch.optim as optim

# Initialize the model, loss function, and optimizer
model = DiabetesModel()
criterion = nn.MSELoss() # MSE loss function
optimiser = optim.Adam(model.parameters(), lr=0.001)

Now we can train the model. Again, the logistic regression tutorial (6.01) may help you undertstand this:

In [8]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training loop (example - you'll likely want to add more epochs)
epochs = 100 # 100 epochs

for epoch in range(epochs):
  # use the train_loader to pass the inputs (x) and targets (y)
  for inputs, targets in train_loader:
    # pass to the GPU (hopefully)
    inputs, targets = inputs.to(device), targets.to(device)

    # pass model to GPU as well
    model.to(device)

    model.train() # put the model object in train mode
    optimiser.zero_grad() # reset the gradiants
    outputs = model(inputs) # create outputs
    loss = criterion(outputs, targets) # compare with Y to get loss
    loss.backward() # backpropogate the loss (next week)
    optimiser.step() # # update the parameters based on this round of training

  # every 10 steps we will print out the current loss
    if (epoch+1) % 10 == 0: # modular arithmetic
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

Epoch [10/100], Loss: 32574.709
Epoch [10/100], Loss: 30733.1797
Epoch [10/100], Loss: 30101.1387
Epoch [10/100], Loss: 24634.5176
Epoch [10/100], Loss: 27176.3789
Epoch [10/100], Loss: 26886.8496
Epoch [10/100], Loss: 34469.9102
Epoch [10/100], Loss: 41769.2969
Epoch [20/100], Loss: 26545.7969
Epoch [20/100], Loss: 25804.5977
Epoch [20/100], Loss: 30798.2988
Epoch [20/100], Loss: 30424.4746
Epoch [20/100], Loss: 27820.0742
Epoch [20/100], Loss: 35334.5703
Epoch [20/100], Loss: 29651.3516
Epoch [20/100], Loss: 33343.0312
Epoch [30/100], Loss: 32411.0742
Epoch [30/100], Loss: 24248.2988
Epoch [30/100], Loss: 27267.1523
Epoch [30/100], Loss: 33051.3945
Epoch [30/100], Loss: 28442.1094
Epoch [30/100], Loss: 25599.1992
Epoch [30/100], Loss: 34458.4375
Epoch [30/100], Loss: 30282.4062
Epoch [40/100], Loss: 29877.4648
Epoch [40/100], Loss: 24701.625
Epoch [40/100], Loss: 27910.7344
Epoch [40/100], Loss: 28448.9551
Epoch [40/100], Loss: 30719.4492
Epoch [40/100], Loss: 32153.75
Epoch [40/100]

We can see loss is significantly lower at the end than it was at the start. However, it is also bouncing around a little still which suggests the model needs more training (100 epochs is not a lot in deep learning terms). However, let's evaluate as before:

In [9]:
# Evaluation (example)
model.eval() # testing mode
mse_values = [] # collect the MSE scores

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs) # predict the test data

        # Calculate Mean Squared Error
        mse = criterion(outputs, targets) # calcualte mse for the batch
        mse_values.append(mse.item()) # add to the list of MSE values

# Calculate and print the average MSE
avg_mse = np.mean(mse_values)
print(f"Average MSE on test set: {avg_mse}")

Average MSE on test set: 19770.9296875


MSE looks expected given training (no obvious sign of overfitting). However, we probably can get better results with tuning and more epochs.

Let's run the loop again a little differently to collect the predicted values (y_hat) and actuals (y) and add them to a dataset for comparions:

In [10]:
# Evaluation
model.eval()
predictions = []
actuals = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predictions.extend(outputs.cpu().numpy())
        actuals.extend(targets.cpu().numpy())

# Create DataFrame
results_df = pd.DataFrame({'Predicted': np.array(predictions).flatten(), 'Actual': np.array(actuals).flatten()})
results_df

,Predicted,Actual
0,25.326241,219.0
1,23.823101,70.0
2,24.763613,202.0
3,30.432854,230.0
4,24.106674,111.0
...,...,...
84,21.683880,153.0
85,20.139679,98.0
86,18.089302,37.0
87,18.887011,63.0


Side-by-side, they don't look great. Can you improve them?

<br><br>

## EXERCISE #1
Try increasing the number of epochs to 1,000 (when the model is fairly well trained then the results printed for each 10x epochs will be fairly stable and not change much). Does this give better results?

<br><br>

## EXERCISE #2 (optional)
Try experimenting with the architecture (number of neurons and/or number of layers). Can we reach an optimal architecture?

##Exercise 1

In [11]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training loop (example - you'll likely want to add more epochs)
epochs = 1000 # 100 epochs

for epoch in range(epochs):
  # use the train_loader to pass the inputs (x) and targets (y)
  for inputs, targets in train_loader:
    # pass to the GPU (hopefully)
    inputs, targets = inputs.to(device), targets.to(device)

    # pass model to GPU as well
    model.to(device)

    model.train() # put the model object in train mode
    optimiser.zero_grad() # reset the gradiants
    outputs = model(inputs) # create outputs
    loss = criterion(outputs, targets) # compare with Y to get loss
    loss.backward() # backpropogate the loss (next week)
    optimiser.step() # # update the parameters based on this round of training

  # every 10 steps we will print out the current loss
    if (epoch+1) % 10 == 0: # modular arithmetic
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

Epoch [10/1000], Loss: 23004.7051
Epoch [10/1000], Loss: 17448.9648
Epoch [10/1000], Loss: 28006.4941
Epoch [10/1000], Loss: 16823.543
Epoch [10/1000], Loss: 18631.5977
Epoch [10/1000], Loss: 19130.5352
Epoch [10/1000], Loss: 23241.7422
Epoch [10/1000], Loss: 32692.1875
Epoch [20/1000], Loss: 13852.082
Epoch [20/1000], Loss: 20992.5703
Epoch [20/1000], Loss: 20680.8008
Epoch [20/1000], Loss: 17468.709
Epoch [20/1000], Loss: 16627.5527
Epoch [20/1000], Loss: 19390.5527
Epoch [20/1000], Loss: 24585.0586
Epoch [20/1000], Loss: 11597.6426
Epoch [30/1000], Loss: 13491.6787
Epoch [30/1000], Loss: 17980.9121
Epoch [30/1000], Loss: 19497.0254
Epoch [30/1000], Loss: 16868.4688
Epoch [30/1000], Loss: 15829.2949
Epoch [30/1000], Loss: 15420.1572
Epoch [30/1000], Loss: 20561.2949
Epoch [30/1000], Loss: 4988.1152
Epoch [40/1000], Loss: 13655.3184
Epoch [40/1000], Loss: 13156.7109
Epoch [40/1000], Loss: 14199.2617
Epoch [40/1000], Loss: 15075.166
Epoch [40/1000], Loss: 17939.6504
Epoch [40/1000], Lo

In [12]:
# Evaluation (example)
model.eval() # testing mode
mse_values = [] # collect the MSE scores

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs) # predict the test data

        # Calculate Mean Squared Error
        mse = criterion(outputs, targets) # calcualte mse for the batch
        mse_values.append(mse.item()) # add to the list of MSE values

# Calculate and print the average MSE
avg_mse = np.mean(mse_values)
print(f"Average MSE on test set: {avg_mse}")

Average MSE on test set: 2864.099609375


In [13]:
# Evaluation
model.eval()
predictions = []
actuals = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predictions.extend(outputs.cpu().numpy())
        actuals.extend(targets.cpu().numpy())

# Create DataFrame
results_df = pd.DataFrame({'Predicted': np.array(predictions).flatten(), 'Actual': np.array(actuals).flatten()})
results_df

,Predicted,Actual
0,147.721497,219.0
1,173.923248,70.0
2,142.954132,202.0
3,291.868988,230.0
4,129.625214,111.0
...,...,...
84,117.279305,153.0
85,89.212601,98.0
86,75.481018,37.0
87,66.589035,63.0


##Exercise 2

In [19]:
import torch
import torch.nn as nn

# Define the model
class DiabetesModel(nn.Module):
    def __init__(self):
        super(DiabetesModel, self).__init__()
        # we'll set up the layers as a sequence using nn.Sequential
        self.layers = nn.Sequential(

            # first layer will be a linear layer that has 5x neurons
            # (5x sets of linear regression)
            # the layer takes the 10 features as input (i.e. 10, 10)
            nn.Linear(10, 10),

            nn.ReLU(), # ReLU activation

            # second linear layer again has 10 neurons
            # this time taking the input as the output of the last layer
            # (which had 5x neurons)
            nn.Linear(10, 10),

            nn.ReLU(), # ReLU again

            # last linear layer takes the output from the previous 10 neurons
            # this time its a single output with no activation
            # i.e. this is the predicitons (regression)
            nn.Linear(10, 1)
        )

    def forward(self, x):
        return self.layers(x) # pass the data through the layers

In [15]:
import torch.optim as optim

# Initialize the model, loss function, and optimizer
model = DiabetesModel()
criterion = nn.MSELoss() # MSE loss function
optimiser = optim.Adam(model.parameters(), lr=0.001)

In [21]:
import torch.optim as optim

# 重新实例化模型和优化器，确保最新的模型层定义（10 -> 10）生效
model = DiabetesModel()
criterion = nn.MSELoss()
optimiser = optim.Adam(model.parameters(), lr=0.001)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Training loop
epochs = 1000

for epoch in range(epochs):
  for inputs, targets in train_loader:
    inputs, targets = inputs.to(device), targets.to(device)

    model.train() # put the model object in train mode
    optimiser.zero_grad() # reset the gradients
    outputs = model(inputs) # create outputs
    loss = criterion(outputs, targets) # compare with Y to get loss
    loss.backward() # backpropagate the loss
    optimiser.step() # update the parameters

  # every 10 epochs we will print out the current loss
  if (epoch+1) % 10 == 0:
    print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

Epoch [10/1000], Loss: 43567.25
Epoch [20/1000], Loss: 50434.6875
Epoch [30/1000], Loss: 45561.1172
Epoch [40/1000], Loss: 44415.7344
Epoch [50/1000], Loss: 33190.9609
Epoch [60/1000], Loss: 9849.5215
Epoch [70/1000], Loss: 23517.2715
Epoch [80/1000], Loss: 5858.1797
Epoch [90/1000], Loss: 8272.1895
Epoch [100/1000], Loss: 10699.0527
Epoch [110/1000], Loss: 3058.4824
Epoch [120/1000], Loss: 9664.5605
Epoch [130/1000], Loss: 3113.825
Epoch [140/1000], Loss: 2663.9033
Epoch [150/1000], Loss: 10076.3965
Epoch [160/1000], Loss: 8500.9023
Epoch [170/1000], Loss: 4147.0811
Epoch [180/1000], Loss: 8850.6826
Epoch [190/1000], Loss: 3609.1406
Epoch [200/1000], Loss: 5855.3589
Epoch [210/1000], Loss: 4977.7979
Epoch [220/1000], Loss: 973.8074
Epoch [230/1000], Loss: 2298.0212
Epoch [240/1000], Loss: 2045.999
Epoch [250/1000], Loss: 4721.9424
Epoch [260/1000], Loss: 787.7705
Epoch [270/1000], Loss: 6215.6968
Epoch [280/1000], Loss: 3074.4082
Epoch [290/1000], Loss: 3135.6724
Epoch [300/1000], Los

In [22]:
# Evaluation (example)
model.eval() # testing mode
mse_values = [] # collect the MSE scores

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs) # predict the test data

        # Calculate Mean Squared Error
        mse = criterion(outputs, targets) # calcualte mse for the batch
        mse_values.append(mse.item()) # add to the list of MSE values

# Calculate and print the average MSE
avg_mse = np.mean(mse_values)
print(f"Average MSE on test set: {avg_mse}")

Average MSE on test set: 2861.2088623046875
